In [ ]:
import numpy as np
import re

def loadDataSet():
    postingList=[['my', 'dog', 'has', 'flea', 'problems', 'help', 'please'],
                 ['maybe', 'not', 'take', 'him', 'to', 'dog', 'park', 'stupid'],
                 ['my', 'dalmation', 'is', 'so', 'cute', 'I', 'love', 'him'],
                 ['stop', 'posting', 'stupid', 'worthless', 'garbage'],
                    ['mr', 'licks', 'ate', 'my', 'steak', 'how', 'to', 'stop', 'him'],
                    ['quit', 'buying', 'worthless', 'dog', 'food', 'stupid']]
    
    classVec = [0,1,0,1,0,1]    # 1 is abusive, 0 not
    return postingList,classVec

def createVocabList(dataSet):
    vocabSet = set([])  # create empty set
    for document in dataSet:
        vocabSet = vocabSet | set(document) # union of the two sets
    return list(vocabSet)

def setOfWords2Vec(vocabList, inputSet):
    returnVec = [0]*len(vocabList)
    for word in inputSet:
        if word in vocabList:
            returnVec[vocabList.index(word)] = 1
        else: print("the word: %s is not in my Vocabulary!" % word)
    return returnVec

def bagOfWords2Vec(vocabList, inputSet):
    returnVec = [0]*len(vocabList)
    for word in inputSet:
        if word in vocabList:
            returnVec[vocabList.index(word)] += 1
    return returnVec

def trainNb0(trainMatrix,trainCategory):
    numTrainDocs = len(trainMatrix)
    numWords = len(trainMatrix[0])
    pAbusive = sum(trainCategory)/(numTrainDocs)
    p0Num = np.ones(numWords); p1Num = np.ones(numWords)      #change to all 1
    p0Denom = 2.0; p1Denom = 2.0                        #change to 2.0
    for i in range(numTrainDocs):
        if trainCategory[i] == 1:
            p1Num += trainMatrix[i]
            p1Denom += sum(trainMatrix[i])
        else:
            p0Num += trainMatrix[i]
            p0Denom += sum(trainMatrix[i])
    p1Vect =np.log(p1Num/p1Denom)        #change to log()
    p0Vect =np.log(p0Num/p0Denom)        #change to log()
    return p0Vect,p1Vect,pAbusive


def classifyNB(vec2Classify, p0Vec, p1Vec, pClass1):
    p1=sum(vec2Classify * p1Vec) + np.log(pClass1)    #element-wise multiply
    p0=sum(vec2Classify * p0Vec) + np.log(1.0 - pClass1)
    if p1 > p0:
        return 1
    else: return 0  

def testingNB():
    listOPosts,listClasses = loadDataSet()
    myVocabList = createVocabList(listOPosts)
    trainMat=[]
    for postinDoc in listOPosts:
        trainMat.append(setOfWords2Vec(myVocabList, postinDoc))
    p0V,p1V,pAb = trainNb0(trainMat,listClasses)
    testEntry = ['love', 'my', 'dalmation']
    thisDoc = np.array(setOfWords2Vec(myVocabList, testEntry))
    print(testEntry,'classified as: ', classifyNB(thisDoc, p0V, p1V, pAb))
    testEntry = ['stupid', 'garbage']
    thisDoc = np.array(setOfWords2Vec(myVocabList, testEntry))
    print(testEntry,'classified as: ', classifyNB(thisDoc, p0V, p1V, pAb))

def textParse(bigString):
    listOfTokens = re.split(r'\W*', bigString)
    return [tok.lower() for tok in listOfTokens if len(tok) > 2]

def spamTest():
    docList=[]; classList = []; fullText =[]
    for i in range(1,26):
        with open('../dataset/Ch04-NaiveBayes/email/spam/%d.txt' % i, 'r', encoding='latin-1') as f:
            wordList = textParse(f.read())
        docList.append(wordList)
        fullText.extend(wordList)
        classList.append(1)
        with open('../dataset/Ch04-NaiveBayes/email/ham/%d.txt' % i, 'r', encoding='latin-1') as f:
            wordList = textParse(f.read())
        docList.append(wordList)
        fullText.extend(wordList)
        classList.append(0)
    vocabList = createVocabList(docList)
    trainingSet = list(range(50))
    testSet=[]
    for i in range(10):
        randIndex = int(np.random.uniform(0,len(trainingSet)))
        testSet.append(trainingSet[randIndex])
        del(trainingSet[randIndex])

    trainMat=[]; trainClasses = []
    
    for docIndex in trainingSet:
        trainMat.append(setOfWords2Vec(vocabList, docList[docIndex]))
        trainClasses.append(classList[docIndex])
    p0V,p1V,pSpam = trainNb0(trainMat,trainClasses)
    errorCount = 0
    for docIndex in testSet:
        wordVector = setOfWords2Vec(vocabList, docList[docIndex])
        if classifyNB(np.array(wordVector),p0V,p1V,pSpam) != classList[docIndex]:
            errorCount += 1
            print("classification error",docList[docIndex])
    print('the error rate is: ',float(errorCount)/len(testSet))
    

In [12]:
listOPosts, listClasses = loadDataSet()
myVocabList = createVocabList(listOPosts)
print(myVocabList)

['ate', 'licks', 'dalmation', 'stupid', 'dog', 'so', 'stop', 'worthless', 'steak', 'mr', 'posting', 'buying', 'please', 'quit', 'help', 'food', 'take', 'love', 'not', 'flea', 'maybe', 'cute', 'I', 'him', 'problems', 'my', 'garbage', 'how', 'park', 'is', 'to', 'has']


In [13]:
print(setOfWords2Vec(myVocabList, listOPosts[0]))
print(setOfWords2Vec(myVocabList, listOPosts[3]))

[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1]
[0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]


In [14]:
trainMat = []
for postinDoc in listOPosts:
    trainMat.append(setOfWords2Vec(myVocabList, postinDoc))
print(trainMat)

p0V,p1V,pAb = trainNb0(trainMat,listClasses)

[[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1], [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0], [0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0], [0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0], [0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]


In [15]:
print(pAb)
print(p0V)
print(p1V)

0.5
[0.04166667 0.04166667 0.04166667 0.         0.04166667 0.04166667
 0.04166667 0.         0.04166667 0.04166667 0.         0.
 0.04166667 0.         0.04166667 0.         0.         0.04166667
 0.         0.04166667 0.         0.04166667 0.04166667 0.08333333
 0.04166667 0.125      0.         0.04166667 0.         0.04166667
 0.04166667 0.04166667]
[0.         0.         0.         0.15789474 0.10526316 0.
 0.05263158 0.10526316 0.         0.         0.05263158 0.05263158
 0.         0.05263158 0.         0.05263158 0.05263158 0.
 0.05263158 0.         0.05263158 0.         0.         0.05263158
 0.         0.         0.05263158 0.         0.05263158 0.
 0.05263158 0.        ]


In [17]:
testingNB()

['love', 'my', 'dalmation'] classified as:  0
['stupid', 'garbage'] classified as:  1


In [19]:
mySent = 'This book is the best book on Python or M.L.I have ever laid eyes upon.'
print(mySent.split())


['This', 'book', 'is', 'the', 'best', 'book', 'on', 'Python', 'or', 'M.L.I', 'have', 'ever', 'laid', 'eyes', 'upon.']


In [26]:
import re
regEx = re.compile('\\W+')
listOfTokens = regEx.split(mySent)
print(listOfTokens)

['This', 'book', 'is', 'the', 'best', 'book', 'on', 'Python', 'or', 'M', 'L', 'I', 'have', 'ever', 'laid', 'eyes', 'upon', '']


In [29]:
print([tok.lower() for tok in listOfTokens if len(tok) > 0])

['this', 'book', 'is', 'the', 'best', 'book', 'on', 'python', 'or', 'm', 'l', 'i', 'have', 'ever', 'laid', 'eyes', 'upon']


In [38]:
path='../dataset/Ch04-NaiveBayes/email/ham/6.txt'
with open(path, 'r', encoding='latin-1') as f:
    emailText = f.read()
listOfTokens = regEx.split(emailText)

In [48]:
spamTest()

classification error []
classification error []
classification error []
classification error []
classification error []
classification error []
classification error []
the error rate is:  0.7


In [52]:
import feedparser

ny=feedparser.parse('https://newyork.craigslist.org/search/stp?format=rss')
print(ny['entries'])
len(ny['entries'])

[]


0